<a href="https://colab.research.google.com/github/adidror005/Sefaria/blob/main/Rambam_RAG_with_LLama_Index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Misc Code For Visualization

In [44]:
import base64
from IPython.display import Image, display

def mm_ink(graph: str):
    graphbytes = graph.encode("utf-8")
    base64_string = base64.urlsafe_b64encode(graphbytes).decode("ascii")
    return "https://mermaid.ink/img/" + base64_string

def mm(graph: str):
    display(Image(url=mm_ink(graph)))

# Build a Rambam Q&A Chatbot with LlamaIndex + OpenAI + Gradio

In this notebook, we build a source-grounded Q&A chatbot over Rambam’s *Mishneh Torah*.

A separate video/notebook covers how to collect and structure the data from the Sefaria API. Here, we assume that step is already done and that we have a DataFrame `df`.


# What We’re Building

We want a chatbot that can answer questions like:

- What does Rambam say about repentance?
- What are the laws of charity?
- How does Rambam describe free will?

The difference between this and ChatGPT is we explicitly find the most relevant sources in the text and then our LLM must use those sources to give its answer.


# Big Picture: What Is RAG?

RAG stands for **Retrieval-Augmented Generation**.

Instead of asking the LLM to answer from memory:

1. Retrieve relevant sources based on semantic similarity (embedding similarity with query) or exact match filtering
2. Give those sources to the LLM
3. Generate an answer grounded in those sources

The key idea is:

> Retrieve first, generate second answer second.

# Tools we will use


- **LlamaIndex** for indexing and retrieval  
  - **OpenAI API** for embeddings and answer generation  
- **Gradio** for the chat interface  

---

In [45]:
# @title
from IPython.display import Markdown,display
display(Markdown("# RAG Flow for Rambam Q&A"))

mm("""
flowchart TD
    A["User Question<br/>What does Rambam say about repentance?"]
    A --> B["Embed Query<br/>q = [0.82, -0.14, 0.33, 0.71]"]
    B --> C["Vector Search<br/>Find K Most Similar Chunks"]

    D1["Rambam Chunk 1<br/>Hilchot Teshuva 2:1"]
    D2["Rambam Chunk 2<br/>Hilchot Deot 1:3"]
    D3["Rambam Chunk 3<br/>Hilchot Teshuva 7:4"]

    D1 --> C
    D2 --> C
    D3 --> C

    C --> E["Return Top K Matches"]
    E --> F["Prompt to LLM<br/>Question + Sources"]
    F --> G["Final Answer<br/>with citations"]
""")

# RAG Flow for Rambam Q&A

# Note on Re-rankers

The vector search step usually retrieves the **top K most similar chunks** based on embedding similarity.

A re-ranker is an optional extra step after retrieval.

Instead of sending all retrieved chunks directly to the LLM, we can do:

1. Retrieve a larger set of candidates, for example top 10 or top 20  
2. Use a re-ranker to score which chunks are truly most relevant  
3. Send only the best few chunks to the LLM

So the basic flow is:

```text
User Question → Vector Search → Top K Chunks → LLM Answer
```

With a re-ranker:

```text
User Question → Vector Search → Candidate Chunks → Re-ranker → Best Chunks → LLM Answer
```

This can improve answer quality when many chunks are semantically similar but only a few directly answer the question.

## 📊 Assumed Input DataFrame

We assume `df` contains **one row per atomic unit** from *Mishneh Torah*.

In most cases, this is a **single halacha**, but introductory and special sections are also included and handled via metadata.

---

### Expected Columns

- `ref` — full textual reference  
- `section_type` — `"halacha"` or `"intro"`
- `section name` — section within the sefer  
- `chapter` — chapter number (NaN for intros)  
- `number` — Halacha number (mitzah/intro number for intros)


---

### 🧠 Notes on Structure

- **Standard halachot:**
  - Have `chapter` and `halacha`
  - Represent the **main retrieval units**

- **Intro / special sections:**
  - Do not follow the standard hierarchy
  - Have `section_type = "intro"`

👉 In RAG, these are **not treated differently structurally** —  
they are handled via **metadata filtering and routing**.

---

### 📌 Load DataFrame


In [29]:
import pandas as pd
df = pd.read_parquet("rambam_df.parquet").fillna(0)


## 📖 Why Mishneh Torah Works Well for RAG

*Mishneh Torah* is highly structured:

1. **Section)**  
2. **Chapter**  
3. **Halacha (Law)**  

Each halacha is:
- Short  
- Self-contained  
- Semantically clear  

This makes it a natural **retrieval unit**.

At the same time, the hierarchy provides rich metadata:
- `sefer`
- `hilchot`
- `chapter`
- `halacha`
- `section_type`

This enables:
- Filtering (e.g., only certain domains)
- Better ranking
- Clear source attribution

---

## 🎯 Key Idea

> Each row in the DataFrame becomes a retrieval unit, and the metadata preserves the full structure of Mishneh Torah — including introductions — allowing flexible and precise RAG behavior.



# Installs Needed
* Maybe need more if not using Colab


In [30]:
!pip install llama_index gradio

#  What is LlamaIndex

* https://developers.llamaindex.ai/python/framework/

Think of LlamaIndex as the **data and retrieval layer** between your Rambam texts and the LLM.

It helps us:

- load text into `Document` objects
- attach metadata
- chunk text into nodes
- create embeddings
- retrieve relevant sources
- optionally re-rank results
- send the best context to the LLM

The LLM still generates the answer, but LlamaIndex helps it use our Rambam data intelligently.

## Documents and Nodes in Llama Index

A `Document` is usually the larger input object we load into LlamaIndex.

LlamaIndex can then split each document into smaller searchable pieces called **Nodes**.

Think of it this way:

- `Document` = original text object we load
- `Node` = smaller chunk used for embeddings and retrieval
- Both can contain metadata such as `ref`, `book`, `hilchot`, `chapter`, and `halacha`

For Rambam, one halacha per document/node is often a strong strategy because retrieved results map cleanly to precise sources.


# Load OpenAI API Keys

In [31]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')


## Configure OpenAI + LlamaIndex

We use OpenAI in two places:

1. **Embeddings** — turning Rambam text and user questions into vectors  
2. **LLM responses** — generating the final answer from retrieved sources

The defaults are pretty good, but I find that we are much better off using the larger embedding model!!!

In [32]:
from llama_index.llms.openai import OpenAI as LlamaOpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings



Settings.llm = LlamaOpenAI(model="gpt-4o-mini", temperature=0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-large")

## Convert DataFrame Rows into LlamaIndex Documents

Each row in the dataframe becomes a `Document`.

The `text` field becomes the content.

The other fields become metadata, which lets us cite sources later.

**CODE MYSELF**

In [33]:
from llama_index.core import Document
import pandas as pd # Import pandas to handle NA values

def df_to_documents(df):
    documents = []

    for row in df.itertuples(index=False):
        # Safely convert row.number to a standard Python int or 0 if NA
        number_val = 0
        if pd.notna(row.number):
            # Convert numpy.int64 to standard int
            number_val = int(row.number)

        metadata = {
            "ref": row.ref,
            "section_name": row.section,
            "section_type": row.section_type,
            "number": number_val # Use the safely converted number_val or 0

        }

        documents.append(
            Document(
                text=row.text,
                metadata=metadata
            )
        )

    return documents

documents = df_to_documents(df)

len(documents), documents[0]

(15741,
 Document(id_='f3c2a812-3c57-49f6-a3a7-0401a5938122', embedding=None, metadata={'ref': 'Mishneh Torah, Transmission of the Oral Law 1', 'section_name': 'Transmission of the Oral Law', 'section_type': 'intro', 'number': 1}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='<b>The Rambam\'s Introduction</b><sup class="footnote-marker">1</sup><i class="footnote">The heading "Introduction" is not found in any of the manuscript editions of the <i>Mishneh Torah</i> and appears to be a printer\'s addition. Note <i>Hilchot Shechitah</i> 1:4, where the Rambam refers to "...the Oral Law, which is called `the mitzvah,\' as we explained in the beginning of this text."<br>By referring to these passages as "the beginning" of the text and not "the introduction to the text," the Rambam implies that the subject matter contained in these passa

In [6]:
df.head(2)

,ref,section,section_type,chapter,number,text
0,"Mishneh Torah, Transmission of the Oral Law 1",Transmission of the Oral Law,intro,<NA>,1,"<b>The Rambam's Introduction</b><sup class=""fo..."
1,"Mishneh Torah, Transmission of the Oral Law 2",Transmission of the Oral Law,intro,<NA>,2,"Moses, our teacher, personally transcribed the..."


## Build the Vector Index

When we call `VectorStoreIndex.from_documents(...)`, LlamaIndex creates embeddings and stores them in a searchable index.

This is where our Rambam dataframe becomes semantic search.


**CODE MYSELF**

In [34]:
from llama_index.core import VectorStoreIndex
index = VectorStoreIndex.from_documents(
    documents,
    show_progress=True
)


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/2048 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Optional, but recommended to Save Embeddings/Indexing for Later Querying. Should Save to VectorStore Like ChromaDB. Llama Index has tools for all this

## Save and Load the Index

Persisting the index prevents us from rebuilding embeddings every time.


```
index.storage_context.persist(persist_dir="./storage")
```
Load later
```
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(
    persist_dir="./storage"
)

index = load_index_from_storage(storage_context)
```

### Let's see a sample embedding

In [11]:
vs = index.storage_context.vector_store
sample = next(iter(vs.data.embedding_dict.values()))
sample

[0.01496124267578125,
 -0.02886962890625,
 -0.0100555419921875,
 0.04302978515625,
 0.0103759765625,
 -0.03662109375,
 -0.0233917236328125,
 0.0025691986083984375,
 -0.00904083251953125,
 0.036376953125,
 -0.0006346702575683594,
 0.049530029296875,
 0.042877197265625,
 -0.00511932373046875,
 0.0288848876953125,
 0.0191192626953125,
 -0.0116119384765625,
 -0.01171875,
 -0.03350830078125,
 0.008392333984375,
 0.01342010498046875,
 0.01076507568359375,
 0.002712249755859375,
 0.03985595703125,
 -0.0207061767578125,
 -0.015411376953125,
 -0.002716064453125,
 0.0010614395141601562,
 0.027801513671875,
 0.0135040283203125,
 0.01568603515625,
 0.0279693603515625,
 0.00402069091796875,
 -0.04193115234375,
 -0.034423828125,
 -0.01206207275390625,
 -0.004852294921875,
 0.03961181640625,
 -0.0264129638671875,
 0.053497314453125,
 -0.005008697509765625,
 0.0242156982421875,
 0.01100921630859375,
 -0.03558349609375,
 -0.01493072509765625,
 0.040618896484375,
 0.02203369140625,
 -0.0190887451171875,

## 🧠 Query Engine vs Retriever (LLama Index Core Components)

### 🔹 Query Engine — “Ask → Answer”

The **query engine** is the high-level interface that produces the final answer.

It handles:

1. Taking retrieved Rambam sources
2. Building a prompt
3. Sending it to the LLM
4. Returning the response



**CODE MYSELF**

In [12]:
query_engine = index.as_query_engine()

response = query_engine.query(
    "What does Rambam say about repentance?"
)

In [13]:
response

Response(response='Rambam emphasizes that while certain transgressions may hinder the process of repentance, they do not completely prevent it. If an individual who has committed such transgressions repents, they are considered a Baal-Teshuvah and are granted a portion in the world to come. He also highlights the importance of free choice, encouraging individuals to strive for repentance and to verbally confess their sins, aiming to cleanse themselves and ultimately achieve the status of a Baal-Teshuvah to merit eternal life.', source_nodes=[NodeWithScore(node=TextNode(id_='009ad21c-4000-43f0-a102-e5ccfa2b2102', embedding=None, metadata={'ref': 'Mishneh Torah, Repentance 4:6', 'section_name': 'Repentance', 'section_type': 'halacha', 'number': 6}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='c4f612ec-2436-46e6-88f1-8bfc184559f4', node_type='4', metadata={'ref': 'Mishneh Torah, Repentance 4:6', 'se

👉 Think:

> **Retriever + Prompt + LLM → Final Answer**

---

### ⚙️ Customizing the Query Engine



In [14]:

query_engine = index.as_query_engine(
    similarity_top_k=6
)

response = query_engine.query(
    "What does Rambam say about repentance?"
)


In [15]:
len(response.source_nodes)

6



You control:

* `text_qa_template` → how the answer is written (grounded, cited, etc.)
* `response_mode` → short vs structured answers
* `similarity_top_k` → how many sources are passed in
* `node_postprocessors` → reranking / refinement

👉 This controls **how the answer is generated**

---

## 🔍 Retriever — “Find the Right Sources”

The **retriever** is the search component.

It does:

1. Embed the query
2. Compare it to stored embeddings
3. Return the top K most relevant chunks


In [16]:
retriever = index.as_retriever(similarity_top_k=5)

nodes = retriever.retrieve(
    "What does Rambam say about free will?"
)


In [17]:
nodes

[NodeWithScore(node=TextNode(id_='1217a0c5-9fd0-4a92-b554-255f1902e1a7', embedding=None, metadata={'ref': 'Mishneh Torah, Repentance 5:1', 'section_name': 'Repentance', 'section_type': 'halacha', 'number': 1}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='a28a5c0e-0731-43bd-8992-741ac18841f4', node_type='4', metadata={'ref': 'Mishneh Torah, Repentance 5:1', 'section_name': 'Repentance', 'section_type': 'halacha', 'number': 1}, hash='b2396731bf62210d661e7e3b582f8f5275fe19f2802c08b7236a006ab8a044ca')}, metadata_template='{key}: {value}', metadata_separator='\n', text='Free will is granted to all men. If one desires to turn himself to the path of good and be righteous, the choice is his. Should he desire to turn to the path of evil and be wicked, the choice is his.<br>This is [the intent of] the Torah\'s statement (Genesis 3:22 : "Behold, man has become unique as ourselves, knowing good and evil," i.



👉 Think:

> **A semantic search engine over Rambam**

---

### ⚙️ Customizing the Retriever further

```python
from llama_index.core import MetadataFilters, ExactMatchFilter

retriever = index.as_retriever(
    similarity_top_k=5,
    filters=MetadataFilters(filters=[
        ExactMatchFilter(key="hilchot", value="Repentance")
    ])
)
```

You control:

* how many results (`top_k`)
* which parts of Rambam are searched
* filtering by metadata (e.g. Sefer HaMadda only)

👉 This controls **what information enters the system**

---

## 🎯 Key Insight

> The retriever does **not generate the answer**
> but it **determines what goes into the prompt**

So:

> **Better retrieval → better sources → better answer**

---

## 🧠 Final Mental Model

| Component    | Role                               |
| ------------ | ---------------------------------- |
| Retriever    | Finds relevant Rambam sources      |
| Query Engine | Turns those sources into an answer |

---

Retriever → raw chunks

Query Engine → chunks + LLM answer

# Custom Grounded Answer Function (Instead of using using .query)

Now we manually build the RAG pipeline:

1. Retrieve relevant Rambam sources
2. Insert sources into a prompt
3. Ask the LLM to answer using only those sources
4. Return both the answer and source table


In [18]:
# Step 8 — Grounded answer function
from openai import OpenAI
from llama_index.core.schema import MetadataMode

client = OpenAI()


def retrieve_sources(index, query, k=5):
    retriever = index.as_retriever(similarity_top_k=k)
    nodes = retriever.retrieve(query)

    rows = []

    for node in nodes:
        md = node.metadata or {}

        rows.append({
            "ref": md.get("ref"),
            "hilchot": md.get("hilchot"),
            "chapter": md.get("chapter"),
            "halacha": md.get("halacha"),
            "score": getattr(node, "score", None),
            "text": node.get_content(metadata_mode=MetadataMode.NONE),
        })

    return pd.DataFrame(rows), nodes


def ask_rambam_with_sources(index, question, k=5):
    results_df, nodes = retrieve_sources(index, question, k=k)

    if results_df.empty:
        return {
            "answer": "I could not retrieve any relevant Rambam source.",
            "sources": results_df,
        }

    sources = "\n\n".join(
        f"{row['ref']}\n{row['text']}"
        for _, row in results_df.iterrows()
    )

    prompt = f'''
You are a Rambam assistant.

Answer in clear English.
Use ONLY the sources below.
Cite the references clearly.
Do not invent sources.
If the sources only partially answer the question, say so.

Question:
{question}

Sources:
{sources}
'''

    response = client.chat.completions.create(
        model="gpt-4o",
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ],
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": results_df,
    }


In [19]:
test = ask_rambam_with_sources(
    index,
    "What does Rambam say about free will?",
    k=5
)

print(test["answer"])
test["sources"]


Rambam, in his Mishneh Torah, discusses the concept of free will extensively. He asserts that free will is granted to all individuals, allowing them to choose their path, whether it be good or evil. This is a fundamental principle of the Torah and mitzvot, as it implies that individuals have the autonomy to make their own decisions and are responsible for their actions (Mishneh Torah, Repentance 5:1, 5:3).

Rambam argues against the notion that God decrees whether a person will be righteous or wicked at the time of their creation. Instead, he emphasizes that each person has the potential to choose their path and develop their character traits independently. This autonomy is crucial for the concepts of reward and punishment to be just, as it holds individuals accountable for their actions (Mishneh Torah, Repentance 5:2, 5:4).

Furthermore, Rambam explains that while people may have certain tendencies or be influenced by their environment, these are not deterministic. Individuals have th

,ref,hilchot,chapter,halacha,score,text
0,"Mishneh Torah, Repentance 5:1",None,None,None,0.660999,Free will is granted to all men. If one desire...
1,"Mishneh Torah, Repentance 5:4",None,None,None,0.616308,Were God to decree that an individual would be...
2,"Mishneh Torah, Repentance 5:3",None,None,None,0.605471,This principle is a fundamental concept and a ...
3,"Mishneh Torah, Repentance 5:2",None,None,None,0.601652,A person should not entertain the thesis held ...
4,"Mishneh Torah, Human Dispositions 1:2",None,None,None,0.593783,"Nevertheless, these are merely tendencies, and..."


# Gradio

````markdown
## What is Gradio?

Gradio lets you turn Python functions into simple web apps.

You define a function → Gradio creates a UI → users interact with it in the browser.

---

## Core Idea

```python
input → your Python function → output
````

That’s it.

---

## Chat Example

```python
import gradio as gr

def chat_fn(message, history):
    return f"You said: {message}"

gr.ChatInterface(fn=chat_fn).launch()
```

---

## What This Does

```text
User types message
   ↓
chat_fn(message, history) runs
   ↓
returned string appears in chat
```

---

## Function Inputs

Your function must accept:

* `message` → the latest user input
* `history` → previous messages (conversation state)

```python
def chat_fn(message, history):
    ...
```

---

## Function Output

Usually return a string:

```python
return "Hello"
```

That becomes the bot reply.

---

## How This Fits Your Rambam Bot

```python
def chat_fn(message, history):
    result = ask_rambam_with_sources(message)
    return result["answer"]
```

Flow:

```text
User question
   ↓
RAG pipeline (retrieve + LLM)
   ↓
Answer returned
   ↓
Displayed in chat
```

---

## Adding Sources (optional)

```python
def chat_fn(message, history):
    result = ask_rambam_with_sources(message)

    sources = "\n\n".join(
        f"{row.ref}\n{row.text}"
        for row in result["sources"].itertuples(index=False)
    )

    return f"{result['answer']}\n\n---\nSources:\n\n{sources}"
```

---

## Share the App

```python
gr.ChatInterface(fn=chat_fn).launch(share=True)
```

* Creates a temporary public link
* Otherwise runs locally:

```text
http://127.0.0.1:7860
```

---

## Debug Rule

If something breaks:

```python
def chat_fn(message, history):
    print(message)
    return "test"
```

If this works → your bug is inside your logic (not Gradio).

---

## Mental Model

```text
Gradio = UI wrapper around your function

Textbox in
Textbox out
Your function does everything in between
```

```
```


## Gradio Chat UI

Now we wrap the Q&A function in a simple chat interface.


In [20]:
def chat_interface(message, history):
    result = ask_rambam_with_sources(
        index,
        message,
        k=5
    )

    return result["answer"]

In [21]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat_interface,
    title="Ask the Rambam",
    description="Ask questions about the Rambam's Mishneh Torah using retrieved sources."
)

demo.launch()


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4b5f76eedc061f1236.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Prettier, More Custom Gradio App Version

In [23]:
import os
import pandas as pd
import gradio as gr

# =========================
# Theme / CSS
# =========================
theme = gr.themes.Soft().set(
    body_background_fill="#f8f5ef",
    block_background_fill="#ffffff",
    block_border_width="1px",
    block_border_color="#e5dccb",
    button_primary_background_fill="#7a5c2e",
    button_primary_background_fill_hover="#644a24",
    button_primary_text_color="#ffffff",
    input_background_fill="#fffdf8",
)

css = """
.gradio-container {
    max-width: 1100px !important;
    margin: 0 auto !important;
}

#app-title {
    text-align: center;
    margin-bottom: 0.25rem;
}

#app-subtitle {
    text-align: center;
    color: #6b6257;
    margin-bottom: 1.25rem;
}

.rambam-card {
    border: 1px solid #e5dccb;
    border-radius: 18px;
    padding: 18px;
    background: linear-gradient(180deg, #fffdf8 0%, #faf6ee 100%);
    box-shadow: 0 6px 18px rgba(60, 40, 10, 0.06);
}

footer {display: none !important;}

.gr-button-primary {
    border-radius: 12px !important;
    font-weight: 600 !important;
}

textarea, .gr-text-input, .gr-textbox {
    border-radius: 12px !important;
}

.gr-chatbot {
    border-radius: 16px !important;
    border: 1px solid #e5dccb !important;
}

.source-box {
    font-size: 0.95rem;
    line-height: 1.5;
    color: #3d372f;
}
"""

# =========================
# Helpers
# =========================
def build_sources_markdown(results_df):
    if results_df is None or results_df.empty:
        return ""

    parts = []
    for row in results_df.itertuples(index=False):
        ref = getattr(row, "ref", "Unknown ref")
        text = getattr(row, "text", "")
        parts.append(f"**{ref}**\n{text}")

    return "\n\n---\n### Sources\n" + "\n\n".join(parts)


def retrieve_sources_safe(question, k=5):
    results_df, nodes = retrieve_sources(index,question, k=k)

    if results_df is None:
        results_df = pd.DataFrame(columns=["ref", "text"])

    if "ref" not in results_df.columns:
        results_df["ref"] = ""
    if "text" not in results_df.columns:
        results_df["text"] = ""

    return results_df, nodes


# =========================
# Streaming retrieval + LLM
# =========================
def ask_rambam_with_sources_stream(question, k=5):
    question = str(question).strip()

    results_df, _ = retrieve_sources_safe(question, k=k)

    if results_df.empty:
        context_text = "No Rambam sources were retrieved."
    else:
        context_text = "\n\n".join(
            f"REF: {row.ref}\nTEXT: {row.text}"
            for row in results_df.itertuples(index=False)
        )

    system_prompt = """
You are a helpful assistant answering questions about the Rambam.

Rules:
1. Answer ONLY from the provided Rambam sources.
2. If the sources do not clearly answer the question, say so clearly.
3. Be concise, clear, and faithful to the source text.
4. Do not invent facts, citations, or quotations.
5. Write in English unless the user asks otherwise.
""".strip()

    user_prompt = f"""
Question:
{question}

Retrieved Rambam sources:
{context_text}

Please answer based only on these sources.
""".strip()

    accumulated = ""

    with client.responses.stream(
        model="gpt-4.1",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    ) as stream:
        for event in stream:
            if event.type == "response.output_text.delta":
                delta = event.delta
                accumulated += delta
                yield accumulated

        final_response = stream.get_final_response()

    final_text = getattr(final_response, "output_text", None) or accumulated
    final_text += build_sources_markdown(results_df)
    yield final_text


# =========================
# ChatInterface handler
# IMPORTANT: this must YIELD for streaming
# =========================
def respond(message, history):
    if not str(message).strip():
        yield "Please enter a question."
        return

    try:
        for partial in ask_rambam_with_sources_stream(message, k=5):
            yield partial
    except Exception as e:
        yield f"Error: {e}"


# =========================
# UI
# =========================
with gr.Blocks(theme=theme, css=css) as demo:
    gr.Markdown("# Ask the Rambam", elem_id="app-title")
    gr.Markdown(
        "A source-grounded Rambam search app using Sefaria + semantic search.",
        elem_id="app-subtitle",
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=600,
                show_label=False,
                avatar_images=(None, None),
            )

            gr.ChatInterface(
                fn=respond,          # streaming stays because respond() yields
                chatbot=chatbot,     # custom styled chatbot
            )

        with gr.Column(scale=1):
            gr.Markdown(
                """
<div class="rambam-card source-box">

### Tips
- Ask in plain English
- Try:
  - free will
  - repentance
  - Chol HaMoed work
  - returning lost objects

### Notes
Answers are grounded in Rambam sources.

</div>
                """
            )

demo.launch(
    prevent_thread_lock=True,
    debug=True,
    share=True
)

/tmp/ipykernel_40877/2762750586.py:175: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=css) as demo:
/tmp/ipykernel_40877/2762750586.py:175: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=css) as demo:
/tmp/ipykernel_40877/2762750586.py:184: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_40877/2762750586.py:184: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d013a95448b982ad7f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://d013a95448b982ad7f.gradio.live


## Final Summary

In this notebook, we started from an existing Rambam dataframe and built a RAG chatbot.

The flow was:

1. DataFrame rows → LlamaIndex Documents  
2. Documents → embeddings  
3. Embeddings → vector index  
4. User question → retrieval  
5. Retrieved sources → LLM answer  
6. Gradio → chat interface

This same pattern can be reused for:

- finance documents
- legal documents
- company knowledge bases
- academic papers
- historical archives
- Torah texts
